# BB 지수 — 2026년 8월 7~9일 출마표 실전 런 (v3, RDS 데이터 공백 해소 후 재실행)

이전 v2는 RDS `race_info_of_horse`의 출전마 데이터 공백(이번 주 전체 경주의 약 35% 출전마 누락) 위에서 실행된 결과였습니다. `[data team lead]`에게 문의 후 원인이 확인됐습니다: 원래 SQL의 `JOIN kra_data.horse`가 INNER JOIN이라, `race_info_of_horse`에는 있지만 `kra_data.horse`에 아직 없는 말들을 조용히 걸러내고 있었습니다. `LEFT JOIN`으로 수정한 새 pull로 완전성 체크(horse_count == 실제 행 수)를 통과한 뒤 이 노트북을 처음부터 재실행했습니다.

**v1→v2 버그 수정 내역(entry-row shift(1) 재계산)은 변경 없이 그대로 재사용**했습니다 — day11 문서 참고.

## Section 0 — 데이터 로드 및 완전성 체크

In [ ]:
import sys
sys.path.insert(0, ".")
import pandas as pd
import numpy as np
from pathlib import Path

RAW_CSV = Path("raw_pull_latest.csv")
PARAMS_JSON = Path("rating_model_params_10var.json")

raw = pd.read_csv(RAW_CSV, dtype={"race_id": str}, low_memory=False)
raw["race_date"] = pd.to_datetime(raw["race_date"])

print(f"{len(raw):,} rows, {raw['race_id'].nunique():,} races")
print(f"Date range: {raw['race_date'].min().date()} to {raw['race_date'].max().date()}")
print(f"Regions: {sorted(raw['region'].unique())}")

In [ ]:
# 완전성 체크 -- horse_count(race_info가 아는 총원) vs 실제 pull된 행 수
TARGET_DATES = ["2026-08-07", "2026-08-08", "2026-08-09"]
week_chk = raw[raw["race_date"].dt.strftime("%Y-%m-%d").isin(TARGET_DATES)].copy()

summary = week_chk.groupby(["race_id","race_date"])["horse_count"].first().reset_index()
actual = week_chk.groupby("race_id").size().rename("actual_rows")
summary = summary.merge(actual, on="race_id")
summary["gap"] = summary["horse_count"] - summary["actual_rows"]

print(summary.groupby(summary["race_date"].dt.date)[["horse_count","actual_rows","gap"]].sum())
print()
n_gap = (summary["gap"] > 0).sum()
print(f"공백 있는 경주: {n_gap} / {len(summary)}")
assert n_gap == 0, "여전히 공백이 있습니다 -- 재풀 필요, 채점으로 넘어가면 안 됨"
print("OK -- 이번 주 35개 경주 전부 완전함 (gap == 0)")

## Section 1 — Feature Engineering (raw KRA 필드 → 10개 모델 변수)

In [ ]:
import build_bc_features as bcf

df = raw.copy()
df["is_start"] = (~df["rank"].isin(bcf.NON_START_RANKS)) & df["rank"].notna()
df["is_start"] = df["is_start"].astype(int)
df["is_win"] = (df["rank"] == 1).astype(int)

df = bcf.build_race_speed_z(df)
df = bcf.build_time_windowed_features(df)
df = bcf.build_last4_features(df)

df["WEIGHT"] = df["impost"]
df["POSTPOS"] = df["back_num"]
print(f"BC features built. Shape: {df.shape}")

In [ ]:
import build_tier1_features as t1f

n0 = len(df)
df = t1f.build_trainer_features(df)
assert len(df) == n0
df = t1f.build_career_starts_and_recency(df)
assert len(df) == n0
df = df.drop(columns=["_row_id"], errors="ignore")
print(f"Tier 1 features built (미수정 상태). Row count check: {n0:,} -> {len(df):,}")

## Section 1.5 — v2 버그 픽스: 안 뛴 출전마 피처 재계산 (변경 없이 재사용)

In [ ]:
from entry_features_fix import recompute_entry_features, ENTRY_VARS

df, is_upcoming = recompute_entry_features(df)
print(f"\n이번주 출전마 행: {int(is_upcoming.sum())}개")

### 검증 — 실제 뛴 경주 행이 전혀 변하지 않았는지 확인

In [ ]:
overlap = (df["is_start"]==1) & is_upcoming
print("is_start=1 이면서 동시에 is_upcoming인 행 (0이어야 함):", int(overlap.sum()))
assert overlap.sum() == 0, "안전장치 실패 -- 실제 뛴 경주와 업커밍 엔트리가 겹칩"
print("OK -- 실제 뛴 경주 행과 업커밍 엔트리 행이 명확히 분리되어 있음")

## Section 2 — 프로즈 모델 로드 + 전체 히스토리 채점 (reference snapshot용)

In [ ]:
import horse_rating_10var_v2 as hr

params = hr.load_params(str(PARAMS_JSON))
beta_raw = np.array(params["beta_raw"])
mu = np.array(params["mu"])
sigma = np.array(params["sigma"])
var_list = params["var_list"]
print("Model fit date:", params.get("fit_date"))
print("Variables:", var_list)

In [ ]:
scored = hr.compute_V(df, var_list, beta_raw, mu, sigma)
scored = hr.block_contributions(scored, var_list, hr.BLOCKS_10, beta_raw, mu, sigma)
snapshot = hr.build_reference_snapshot(scored, hr.BLOCKS_10, months=hr.ROLLING_MONTHS,
                                       min_rows=hr.MIN_REFERENCE_ROWS)
print("Reference as_of_date:", snapshot["as_of_date"], "| n =", snapshot["n_reference_rows"])

## Section 3 — 이번주 출전마 로스터 (compute_V 이전 df에서 추출)

**중요**: `compute_V()`는 10개 변수 중 하나라도 NaN인 행을 통계적으로 걸러냅니다 (reference population 계산 목적상 정상 동작). 하지만 이 필터를 그대로 이번 주 로스터에 적용하면 데뷔마(과거 기록 0)가 '정보부족'으로 표시되지 않고 아예 통째로 사라집니다. 그래서 이번 주 로스터는 `compute_V` **이전**의 `df`(Section 1.5에서 만든 `is_upcoming` 플래그 기준)에서 뽑아, 정보부족 말도 명시적으로 표에 남도록 했습니다.

In [ ]:
week_roster = df[is_upcoming].copy()
print(f"이번주 전체 출전마 로스터: {len(week_roster)}두, {week_roster['race_id'].nunique()}개 경주")

## Section 4 — 전체 경주 채점 (정보부족 말은 NaN으로 명시, 0으로 처리하지 않음)

In [ ]:
full_rows, top5_rows, race_meta_rows = [], [], []

for rid in sorted(week_roster["race_id"].unique()):
    race = week_roster[week_roster["race_id"] == rid].copy().sort_values("back_num")
    meta = race.iloc[0]

    ratable, unratable = [], []
    for _, row in race.iterrows():
        rec = {"horse_num": int(row["back_num"]), "horse_id": int(row["horse_id"])}
        if row[var_list].isna().any():
            rec["missing"] = [v for v in var_list if pd.isna(row[v])]
            unratable.append(rec)
        else:
            for v in var_list:
                rec[v] = float(row[v])
            ratable.append(rec)

    n_total, n_rat = len(race), len(ratable)
    race_meta_rows.append({"race_id": rid, "race_date": meta["race_date"].date(),
        "region": meta["region"], "race_class": meta["race_class"],
        "distance": int(meta["distance"]), "n_total": n_total,
        "n_ratable": n_rat, "n_unratable": n_total - n_rat})

    if n_rat < 2:
        print(f"race_id {rid}: 채점 가능한 말 2마리 미만 ({n_rat}마리) -- 전체 정보부족으로 표시")
        for h in unratable:
            full_rows.append({"race_id": rid, "race_date": meta["race_date"].date(),
                "region": meta["region"], "race_class": meta["race_class"],
                "distance": int(meta["distance"]), "horse_num": h["horse_num"],
                "horse_id": h["horse_id"], "ability_score": np.nan, "fund_p": np.nan,
                "status": "정보부족", "missing_vars": ";".join(h["missing"])})
        continue

    if unratable:
        print(f"race_id {rid}: 마번 {[h['horse_num'] for h in unratable]} 정보부족으로 제외 "
              f"(fund_p는 단 {n_rat}마리 사이에서만 합 100%)")

    results = hr.score_race(ratable, var_list, hr.BLOCKS_10, beta_raw, mu, sigma,
                            reference_snapshot=snapshot)

    for r in sorted(results, key=lambda x: -x.ability_score):
        full_rows.append({"race_id": rid, "race_date": meta["race_date"].date(),
            "region": meta["region"], "race_class": meta["race_class"],
            "distance": int(meta["distance"]), "horse_num": r.horse_num,
            "horse_id": r.horse_id, "ability_score": r.ability_score,
            "fund_p": r.win_expectancy, "status": "채점완료", "missing_vars": ""})
    for h in unratable:
        full_rows.append({"race_id": rid, "race_date": meta["race_date"].date(),
            "region": meta["region"], "race_class": meta["race_class"],
            "distance": int(meta["distance"]), "horse_num": h["horse_num"],
            "horse_id": h["horse_id"], "ability_score": np.nan, "fund_p": np.nan,
            "status": "정보부족", "missing_vars": ";".join(h["missing"])})

    for rank, r in enumerate(sorted(results, key=lambda x: -x.ability_score)[:5], 1):
        top5_rows.append({"race_id": rid, "race_date": meta["race_date"].date(),
            "region": meta["region"], "race_class": meta["race_class"],
            "distance": int(meta["distance"]), "top5_rank": rank,
            "horse_num": r.horse_num, "horse_id": r.horse_id,
            "ability_score": r.ability_score, "fund_p": r.win_expectancy,
            "n_unratable_in_race": n_total - n_rat})

full_df = pd.DataFrame(full_rows)
top5_df = pd.DataFrame(top5_rows)
meta_df = pd.DataFrame(race_meta_rows)
print()
print(f"총 {meta_df[meta_df['n_ratable']>=2].shape[0]} / {len(meta_df)}개 경주 채점 완료")
print(f"출전마 총 {len(full_df)}두 | 채점완료 {(full_df['status']=='채점완료').sum()}두 | "
      f"정보부족 {(full_df['status']=='정보부족').sum()}두")

## Section 5 — 경주별 전체 출전마 표

In [ ]:
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 300)

for rid in sorted(full_df["race_id"].unique()):
    race_rows = full_df[full_df["race_id"] == rid].copy()
    meta_row = meta_df[meta_df["race_id"] == rid].iloc[0]
    n_unrat = int(meta_row["n_unratable"])

    header = (f"### {meta_row['race_date']} | {meta_row['region']} {str(rid)[-2:]}R "
              f"| {meta_row['race_class']} {meta_row['distance']}m | race_id {rid}")
    display(Markdown(header))

    if n_unrat > 0:
        display(Markdown(
            f"⚠️ **fund_p 과대평가 플래그**: 이 경주는 정보부족 말이 {n_unrat}마리 있어, "
            f"아래 fund_p는 채점된 {int(meta_row['n_ratable'])}마리 사이에서만 합 100%가 되도록 계산되어 "
            f"실제보다 높게 나옵니다 (BB 지수/순위는 영향 없음)."
        ))

    display_df = race_rows[["horse_num", "ability_score", "fund_p", "status"]].copy()
    display_df["fund_p"] = display_df["fund_p"].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
    display_df["ability_score"] = display_df["ability_score"].apply(lambda x: f"{x:.0f}" if pd.notna(x) else "—")
    display_df.columns = ["마번", "BB 지수", "fund_p", "상태"]
    display(display_df.reset_index(drop=True))
    print()

## Section 5.5 — Top-5 마번 요약 표

In [ ]:
for rid in sorted(top5_df["race_id"].unique()):
    race_rows = top5_df[top5_df["race_id"] == rid].sort_values("top5_rank").copy()
    meta_row = meta_df[meta_df["race_id"] == rid].iloc[0]
    n_unrat = int(race_rows.iloc[0]["n_unratable_in_race"])

    header = (f"### {meta_row['race_date']} | {meta_row['region']} {str(rid)[-2:]}R "
              f"| {meta_row['race_class']} {meta_row['distance']}m")
    display(Markdown(header))

    if n_unrat > 0:
        display(Markdown(
            f"⚠️ **fund_p 과대평가 플래그**: 이 경주는 정보부족 말이 {n_unrat}마리 있어, "
            f"아래 fund_p가 실제보다 높게 나옵니다 (BB 지수/순위는 영향 없음)."
        ))

    display_df = race_rows[["top5_rank", "horse_num", "ability_score", "fund_p"]].copy()
    display_df["fund_p"] = display_df["fund_p"].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
    display_df["ability_score"] = display_df["ability_score"].apply(lambda x: f"{x:.0f}" if pd.notna(x) else "—")
    display_df.columns = ["순위", "마번", "BB 지수", "fund_p"]
    display(display_df.reset_index(drop=True))
    print()

## Section 6 — 결과 저장

In [ ]:
import os
OUT_DIR = Path("outputs")
os.makedirs(OUT_DIR, exist_ok=True)

full_df.to_csv(OUT_DIR / "bb_index_full_v3.csv", index=False)
top5_df.to_csv(OUT_DIR / "bb_index_top5_v3.csv", index=False)
meta_df.to_csv(OUT_DIR / "bb_index_race_coverage_v3.csv", index=False)
print("저장됨:", list(OUT_DIR.glob("*v3.csv")))

## 참고

- v2 대비 변경점: (1) RDS pull SQL의 `kra_data.horse` JOIN을 INNER→LEFT로 수정, (2) 완전성 체크를 Section 0에 추가, (3) 이번 주 로스터를 `compute_V` 이전 시점에서 추출해 데뷔마도 '정보부족'으로 명시 표시되도록 함.
- entry-row shift(1) 재계산 로직(Section 1.5)은 v2에서 변경 없이 그대로 재사용.
- 데뷔마 fund_p 과대평가 이슈는 여전히 open question (day11/day12 문서 참고) — BB 지수 자체에는 영향 없음이 수학적으로 확인됨.